# Cross-conformal predictive systems

This notebook covers both supported targets: non-negative integer counts and continuous values. Each example cross-fits a location and a dispersion model, then uses the resulting predictive distribution for quantiles, cumulative and exceedance probabilities, evaluation, and Newsvendor decisions.

In [1]:
import os
import sys

sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import cross_val_predict, train_test_split

from tinyconformal.distribution import (
    ContinuousCrossConformalPredictiveSystem,
    DiscreteCrossConformalPredictiveSystem,
)
from tinyconformal.utils import FirstStageEvaluator, NewsvendorSolver

pd.set_option("display.max_columns", 20)

## 1. Discrete CPS

The discrete system is intended for ordered integer outcomes such as demand counts. It provides `cdf`, `sf`, `ppf`, and `pmf`; quantiles and optimized quantities are integers.

In [2]:
rng = np.random.default_rng(42)
X_count = rng.uniform(0, 3, size=(1500, 1))
mean_count = np.exp(0.7 + 0.55 * X_count[:, 0])
y_count = rng.poisson(mean_count)
X_count_train, X_count_test, y_count_train, y_count_test = train_test_split(
    X_count, y_count, test_size=0.2, random_state=42
)

count_location = HistGradientBoostingRegressor(
    loss="poisson", max_iter=150, random_state=42
)
count_scale = RandomForestRegressor(
    n_estimators=100, min_samples_leaf=8, random_state=42, n_jobs=-1
)
count_cps = DiscreteCrossConformalPredictiveSystem(
    learner=count_location,
    dispersion_learner=count_scale,
    cv=5,
    n_jobs=-1,
    minimum=0,
).fit(X_count_train, y_count_train)
count_distribution = count_cps.predict_distribution(X_count_test)

### Discrete PPF, CDF, SF, and PMF

A scalar is evaluated for every prediction row, a one-dimensional input defines a common grid, and a two-dimensional input with one row per prediction is evaluated row-wise. For inventory level `N`, `cdf(N)` is the service probability `P(Y<=N)` and `sf(N)` is the exceedance risk `P(Y>N)`.

In [3]:
count_quantiles = count_distribution.ppf([0.50, 0.90, 0.95])
inventory_level = 8
count_summary = pd.DataFrame({
    "y": y_count_test,
    "q50": count_quantiles[:, 0],
    "q90": count_quantiles[:, 1],
    "q95": count_quantiles[:, 2],
    "service_probability": count_distribution.cdf(inventory_level),
    "exceedance_risk": count_distribution.sf(inventory_level),
    "mass_at_8": count_distribution.pmf(inventory_level),
})
count_summary["probability_check"] = (
    count_summary["service_probability"] + count_summary["exceedance_risk"]
)
count_summary.head()

,y,q50,q90,q95,service_probability,exceedance_risk,mass_at_8,probability_check
0,8,3,6,7,0.980849,0.019151,0.019151,1.0
1,1,4,7,8,0.954205,0.045795,0.029142,1.0
2,1,2,5,5,0.996669,0.003331,0.003331,1.0
3,7,9,14,15,0.487094,0.512906,0.118235,1.0
4,5,10,14,16,0.278934,0.721066,0.119067,1.0


In [4]:
display(count_distribution.evaluate(y_count_test, coverages=[0.80, 0.90, 0.95]))
count_distribution.pmf(np.arange(0, 6))[:5]

,coverage,coverage_rate,interval_width_mean,mwis
0,0.80,0.903,6.303,7.770
1,0.90,0.943,8.330,9.730
2,0.95,0.977,9.733,10.667


array([[0.13488759, 0.14487927, 0.1898418 , 0.17985012, 0.13821815,
        0.08409659],
       [0.06827644, 0.0907577 , 0.15653622, 0.16902581, 0.17152373,
        0.12323064],
       [0.10074938, 0.20233139, 0.25145712, 0.22231474, 0.122398  ,
        0.0549542 ],
       [0.0041632 , 0.00666112, 0.00999167, 0.02248127, 0.03996669,
        0.07410491],
       [0.        , 0.00166528, 0.00166528, 0.00249792, 0.01248959,
        0.01915071]])

### First-stage diagnostics

These diagnostics evaluate the location learner independently of conformal scaling. Out-of-fold predictions avoid evaluating the learner on rows used to fit it.

In [18]:
count_oof = cross_val_predict(
    count_location, X_count_train, y_count_train, cv=5, n_jobs=-1
)
count_first_stage = pd.DataFrame({"y": y_count_train, "y_pred": count_oof})
FirstStageEvaluator.calibration_table(count_first_stage, n_bins=10)

,calibration_bin,count,mean_prediction,mean_observed,mean_residual
0,"(1.1190000000000002, 2.093]",121,1.782361,2.181818,0.399457
1,"(2.093, 2.746]",119,2.433578,2.638655,0.205077
2,"(2.746, 3.225]",120,2.976337,3.291667,0.315330
3,"(3.225, 3.739]",121,3.500697,3.355372,-0.145325
4,"(3.739, 4.586]",120,4.168584,4.083333,-0.085251
5,"(4.586, 5.302]",119,4.974395,4.882353,-0.092042
6,"(5.302, 6.473]",120,5.809628,6.008333,0.198705
7,"(6.473, 7.627]",122,7.054457,7.352459,0.298002
8,"(7.627, 9.392]",122,8.608108,8.057377,-0.550731
9,"(9.392, 11.657]",116,10.098928,9.603448,-0.495479


### Optimize discrete inventory

The Newsvendor critical ratio is `underage_cost / (underage_cost + overage_cost)`. The optimizer evaluates the CPS PPF at that probability; marginal benefit uses the discrete CDF to value each additional unit.

In [6]:
count_decisions = pd.DataFrame({
    "unique_id": np.arange(len(y_count_test)).astype(str),
    "ds": pd.Timestamp("2026-01-01"),
    "shortage_cost": 9.0,
    "holding_cost": 1.0,
})
count_plan = NewsvendorSolver.optimize_distribution(
    count_decisions,
    count_distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
)
display(count_plan.head())
NewsvendorSolver.marginal_benefit_distribution(
    count_decisions,
    count_distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
    units=range(0, 11, 2),
).head()

,unique_id,ds,shortage_cost,holding_cost,critical_ratio,y_optimal
0,0,2026-01-01,9.0,1.0,0.9,6.0
1,1,2026-01-01,9.0,1.0,0.9,7.0
2,2,2026-01-01,9.0,1.0,0.9,5.0
3,3,2026-01-01,9.0,1.0,0.9,14.0
4,4,2026-01-01,9.0,1.0,0.9,14.0


,unique_id,ds,shortage_cost,holding_cost,MB(k=0),MB(k=2),MB(k=4),MB(k=6),MB(k=8),MB(k=10)
0,0,2026-01-01,9.0,1.0,9.0,6.202331,2.505412,0.282265,-0.616986,-0.908410
1,1,2026-01-01,9.0,1.0,9.0,7.409659,4.154038,1.206495,-0.250624,-0.783514
2,2,2026-01-01,9.0,1.0,9.0,5.969192,1.231474,-0.542048,-0.933389,-0.983347
3,3,2026-01-01,9.0,1.0,9.0,8.891757,8.567027,7.426311,5.311407,2.946711
4,4,2026-01-01,9.0,1.0,9.0,8.983347,8.941715,8.625312,7.401332,4.828476


## 2. Continuous CPS

The continuous system keeps real-valued support. It provides `cdf`, `sf`, `ppf`, and central intervals, but no PMF.

In [7]:
X_continuous = rng.uniform(0, 10, size=(1500, 1))
y_continuous = (
    20
    + 3 * X_continuous[:, 0]
    + rng.normal(0, 1 + 0.4 * X_continuous[:, 0])
)
X_cont_train, X_cont_test, y_cont_train, y_cont_test = train_test_split(
    X_continuous, y_continuous, test_size=0.2, random_state=42
)

continuous_location = RandomForestRegressor(
    n_estimators=100, min_samples_leaf=8, random_state=42, n_jobs=-1
)
continuous_scale = RandomForestRegressor(
    n_estimators=100, min_samples_leaf=8, random_state=43, n_jobs=-1
)
continuous_cps = ContinuousCrossConformalPredictiveSystem(
    learner=continuous_location,
    dispersion_learner=continuous_scale,
    cv=5,
    n_jobs=-1,
).fit(X_cont_train, y_cont_train)
continuous_distribution = continuous_cps.predict_distribution(X_cont_test)

### Continuous PPF, CDF, SF, and intervals

For an upper threshold, `sf(N)` directly reports `P(Y>N)`. An interval probability can be obtained by subtracting two CDF evaluations.

In [8]:
continuous_quantiles = continuous_distribution.ppf([0.05, 0.50, 0.95])
continuous_summary = pd.DataFrame({
    "y": y_cont_test,
    "q05": continuous_quantiles[:, 0],
    "q50": continuous_quantiles[:, 1],
    "q95": continuous_quantiles[:, 2],
    "P(Y<=30)": continuous_distribution.cdf(30),
    "P(Y>30)": continuous_distribution.sf(30),
})
display(continuous_summary.head())
display(continuous_distribution.evaluate(y_cont_test, coverages=[0.80, 0.90, 0.95]))
continuous_distribution.interval(coverage=0.90)[:5]

,y,q05,q50,q95,P(Y<=30),P(Y>30)
0,22.766250,20.400957,22.519537,24.811690,1.000000,0.000000
1,40.607013,34.413049,45.150659,56.767996,0.009992,0.990008
2,39.513924,32.626110,40.507229,49.034043,0.016653,0.983347
3,29.292038,25.639255,29.341798,33.347688,0.614488,0.385512
4,24.614642,20.796405,22.663651,24.683878,1.000000,0.000000


,coverage,coverage_rate,interval_width_mean,mwis
0,0.80,0.757,7.675,11.433
1,0.90,0.867,10.257,13.685
2,0.95,0.943,13.181,15.357


array([[20.40095708, 24.81169044],
       [34.4130488 , 56.7679957 ],
       [32.62611008, 49.03404251],
       [25.63925526, 33.34768756],
       [20.79640509, 24.68387845]])

### Optimize continuous capacity

The workflow is identical, but the continuous PPF returns a real-valued optimal quantity. Decision costs belong to the row-aligned decision frame; they are not prediction features.

In [9]:
continuous_decisions = pd.DataFrame({
    "unique_id": np.arange(len(y_cont_test)).astype(str),
    "ds": pd.Timestamp("2026-01-01"),
    "shortage_cost": 9.0,
    "excess_cost": 1.0,
})
NewsvendorSolver.optimize_distribution(
    continuous_decisions,
    continuous_distribution,
    underage_cost="shortage_cost",
    overage_cost="excess_cost",
).head()

,unique_id,ds,shortage_cost,excess_cost,critical_ratio,y_optimal
0,0,2026-01-01,9.0,1.0,0.9,24.245284
1,1,2026-01-01,9.0,1.0,0.9,53.897274
2,2,2026-01-01,9.0,1.0,0.9,46.927009
3,3,2026-01-01,9.0,1.0,0.9,32.357805
4,4,2026-01-01,9.0,1.0,0.9,24.184667


## Support comparison

| Target | CDF | SF | PPF | PMF | Newsvendor output |
|---|---:|---:|---:|---:|---|
| Non-negative integer counts | Yes | Yes | Yes | Yes | Integer |
| Continuous values | Yes | Yes | Yes | No | Continuous |

Use the discrete CPS only for genuinely ordered integer outcomes. For nominal labels, use the classifiers in `tinyconformal.classifier`.